## File Versioning in Cloud Storage

# Mastering File Versioning in Cloud Storage

Welcome to the lesson on mastering file versioning and management in cloud storage. **File versioning** is a powerful feature that protects your data by keeping multiple versions of objects. With versioning enabled, every time you upload or modify an object, a new version is created and preserved. This allows you to recover previous versions in case of accidental overwrites or deletions, ensuring data integrity and recoverability.

In this lesson, you will learn how to enable, use, and manage versioning in your cloud storage buckets. These skills are essential for robust backup strategies and applications requiring historical data retention.

---

## Enabling Versioning on a Bucket

To enable versioning, you must update the bucket's configuration. Once active, every new upload or modification results in a new preserved version.

```python
from google.cloud import storage
from google.auth.credentials import AnonymousCredentials
import os

client = storage.Client(
    project=os.environ.get('PROJECT_ID'),
    credentials=AnonymousCredentials(),
    client_options={'api_endpoint': os.environ.get('STORAGE_HOST')}
)

# Get the bucket object
bucket = client.get_bucket('my_bucket')

# Enable versioning
bucket.versioning_enabled = True
bucket.patch()

print('Versioning has been enabled for the bucket.')

```

---

## Suspending Versioning on a Bucket

You cannot completely "disable" versioning once it has been enabled, but you can **suspend** it.

* **Suspending** means new uploads will overwrite the current version without creating a new one.
* **Existing versions** remain accessible and are not removed.

```python
# Suspend versioning on the bucket
bucket.versioning_enabled = False
bucket.patch()

print('Versioning has been suspended for the bucket.')

```

---

## Verifying Versioning Status

After modification, it is important to verify the status by inspecting the bucket's properties. You must use `reload()` to fetch the latest metadata from the server.

```python
# Refresh the bucket's metadata
bucket.reload()

# Check the versioning status
if bucket.versioning_enabled:
    print('Versioning Status: Enabled')
else:
    print('Versioning Status: Suspended')

```

---

## Uploading to Version-Enabled Buckets

Every upload creates a new version if versioning is enabled. You can upload local files or raw strings.

### Uploading a Local File

```python
filename = 'example.txt'
bucket = client.get_bucket('my_bucket')
blob = bucket.blob(filename)

blob.upload_from_filename(filename)
print(f'{filename} has been uploaded.')

```

### Uploading Content Directly (String)

```python
content = "Hello, World!"
blob = bucket.blob('example.txt')
blob.upload_from_string(content)
print('Content has been directly uploaded as a new version.')

```

---

## Retrieving and Downloading Objects

By default, requesting an object without a version identifier retrieves the **latest** version.

```python
# Retrieve the latest version as text
blob = bucket.blob('example.txt')
content = blob.download_as_text()
print('Retrieved latest object version:', content)

# Download the latest version to a local file
blob.download_to_filename('downloaded_example.txt')

```

---

## Listing and Retrieving Specific Versions

To access older data, you need the **Generation Number**, which acts as the unique identifier for that specific version.

### Listing All Versions

```python
# Use versions=True to see the history
blobs = client.list_blobs('my_bucket', versions=True, prefix='example.txt')

for blob in blobs:
    print(f"Name: {blob.name}")
    print(f"Generation: {blob.generation}")
    print(f"Updated: {blob.updated}")
    print(f"Is Latest: {blob.metageneration == 1}\n")

```

### Retrieving a Specific Version

Once you have the generation number, you can instantiate a `Blob` object specifically for that version.

```python
generation_number = '1234567890' # Example generation number
specific_blob = storage.Blob('example.txt', bucket, generation=generation_number)

content = specific_blob.download_as_text()
print('Content of the specific version:', content)

# Download to a file
specific_blob.download_to_filename('version_history_backup.txt')

```

---

## Lesson Summary

In this lesson, you mastered:

* **Enabling and Suspending**: Controlling the versioning toggle.
* **Verification**: Checking status with `reload()`.
* **Creation**: Uploading data to trigger new versions.
* **Retrieval**: Accessing the latest content or historical data via **Generation Numbers**.
* **Auditing**: Listing the full history of an object using `versions=True`.

These capabilities are essential for protecting your data against accidental loss and maintaining a reliable audit trail.

## Exploring Google Cloud Storage Document Versioning

Explore versioning in cloud storage without writing a single line of code. In this task, you'll observe how versions of a document are uploaded to a GCS bucket, with each generation number captured. Understand how Google Cloud Storage versioning maintains every document update. Simply click the Run button to upload document versions and see how to retrieve the latest and a specific earlier version by its generation number. Witness version control in action with just a click!

Important Note: Running scripts can alter the filesystem's state or modify the resources in our GCP simulator. To revert to the initial state, you can use the reset button located in the top right corner. However, keep in mind that resetting will erase any code changes. To preserve your code during a reset, consider copying it to the clipboard.

```python
from google.cloud import storage
from google.auth.credentials import AnonymousCredentials
from google.api_core.exceptions import Conflict
import os

# Initialize the Google Cloud Storage client with emulator settings
client = storage.Client(
    project=os.environ.get('PROJECT_ID'),
    credentials=AnonymousCredentials(),
    client_options={'api_endpoint': os.environ.get('STORAGE_HOST')}
)

# Create the bucket 'cosmo-archive-data' with versioning enabled
bucket_name = 'cosmo-archive-data'
bucket = client.bucket(bucket_name)
try:
    bucket.create()
    print(f"Created bucket: {bucket_name}")
except Conflict:
    print(f"Bucket {bucket_name} already exists")

# Enable versioning for the 'cosmo-archive-data' bucket
bucket.versioning_enabled = True
bucket.patch()
print("Versioning enabled for the bucket")

# Upload version 1 and print its generation number
obj_key = 'versioned_document.txt'
blob_v1 = bucket.blob(obj_key)
blob_v1.upload_from_string('This is the content of version 1.')
print(f"Uploaded version 1 with generation number: {blob_v1.generation}")

# Upload version 2 and print its generation number
blob_v2 = bucket.blob(obj_key)
blob_v2.upload_from_string('This is the content of version 2.')
print(f"Uploaded version 2 with generation number: {blob_v2.generation}")

# Retrieve the latest version of the document without specifying the generation number
latest_blob = bucket.blob(obj_key)
latest_blob.reload()
print(f"Retrieved latest version of 'versioned_document.txt' with generation: {latest_blob.generation}")

# Retrieve and print version 1 using its generation number
specific_blob = bucket.blob(obj_key, generation=blob_v1.generation)
specific_blob.reload()
print(f"Retrieved 'versioned_document.txt' with specific generation number: {blob_v1.generation}")

# Download and print version 1 using its generation number
specific_blob.download_to_filename('version1.txt')

# Now, let's open and print the content of the downloaded file.
with open('version1.txt', 'r') as file:
    content = file.read()
    print(f"Content of version 1: {content}")

# Download the latest version
latest_blob.download_to_filename('latest_version.txt')

# Now, let's open and print the content of the downloaded latest version file.
with open('latest_version.txt', 'r') as file:
    content = file.read()
    print(f"Content of latest version: {content}")

```

This simulation provides a clear view of how Google Cloud Storage handles **non-current** data. By enabling versioning, the bucket stops overwriting data and starts "stacking" it, using **Generation Numbers** as unique timestamps.

### The Lifecycle of a Versioned Object

#### 1. Enabling the "Stack"

When you set `bucket.versioning_enabled = True`, GCS changes its behavior. Instead of the newest file replacing the old one, the old one is preserved in the background. Each file is assigned a `generation` number—a unique integer that identifies that exact state of the file.

#### 2. The Latest Version (Live Object)

When you request `versioned_document.txt` without any extra parameters, GCS always gives you the "Live" version (the most recent upload). In the script, this is Version 2.

#### 3. Time Travel with Generation Numbers

The real power is shown when the script retrieves Version 1. By providing the specific `generation` number to the `blob` constructor:

```python
specific_blob = bucket.blob(obj_key, generation=blob_v1.generation)

```

You are effectively telling GCS to ignore the current file and reach into the archive to pull out the historical data.

---

### Key Observations from the Script

* **Metadata Consistency**: Notice that both versions share the same `name` (`versioned_document.txt`), but their `generation` numbers are distinct.
* **Immutable History**: Once a version is created with a specific generation number, its content cannot be changed. If you "edit" it, GCS simply creates a *third* generation.
* **Protection Against Deletion**: If you were to delete the live version of this document, Version 1 and Version 2 would still exist as "non-current" versions, allowing for easy recovery.

### Summary of Commands

| Action | Detail |
| --- | --- |
| **`bucket.patch()`** | Essential to send the `versioning_enabled` change to the server. |
| **`blob.generation`** | The unique ID created by GCS upon every upload. |
| **`bucket.blob(name, generation=...)`** | The method used to "aim" your client at a specific point in time. |

This mechanism is the backbone of data durability in GCS, ensuring that even a "deleted" or "overwritten" file can be restored as long as you have its generation number.

## Enable Cloud Storage Bucket Versioning

Excellent work, Space Explorer! Now, it's time to solidify your understanding of Cloud Storage versioning. Your space mission depends on mastering data version control for efficient archiving in the cloud. Can you enable versioning on a bucket? This is a crucial step to ensure that every version of your data is secured and retrievable.

Important Note: Running scripts might alter the filesystem's state or modify the resources in our GCP simulator. To revert to the initial state, use the reset button located in the top right corner. However, remember that resetting will erase any code changes. To preserve your code during a reset, consider copying it to the clipboard.

```python
import os
from google.cloud import storage
from google.auth.credentials import AnonymousCredentials

# Initialize Cloud Storage client
client = storage.Client(
    project=os.environ.get('PROJECT_ID'),
    credentials=AnonymousCredentials(),
    client_options={'api_endpoint': os.environ.get('STORAGE_HOST')}
)

bucket_name = 'archive-bucket'
# TODO: Create a new bucket and enable the versioning for it

```

The `google.api_core.exceptions.Conflict` error usually occurs because the bucket name you are trying to use already exists. In Google Cloud Storage, **bucket names are globally unique**—meaning if someone else in the world has a bucket named `archive-bucket`, you cannot create one with that same name.

It can also happen if you've already run the script once; the bucket is already there, so trying to "create" it again causes a conflict.

### The Fix: Check Before You Create

The most robust way to handle this is to check if the bucket exists before attempting the `create()` call.

```python
import os
from google.cloud import storage
from google.auth.credentials import AnonymousCredentials
from google.api_core.exceptions import Conflict

# Initialize Cloud Storage client
client = storage.Client(
    project=os.environ.get('PROJECT_ID'),
    credentials=AnonymousCredentials(),
    client_options={'api_endpoint': os.environ.get('STORAGE_HOST')}
)

# Use a more unique name to avoid global conflicts
bucket_name = 'archive-bucket-unique-12345'
bucket = client.bucket(bucket_name)

# 1. Check if the bucket exists first
if not bucket.exists():
    try:
        bucket.create()
        print(f"Bucket {bucket_name} created.")
    except Conflict:
        # Catching the race condition where someone creates it 
        # in the millisecond between your check and create call
        print(f"Bucket {bucket_name} was just created by someone else.")
else:
    print(f"Bucket {bucket_name} already exists. Moving to configuration.")

# 2. Enable versioning
bucket.versioning_enabled = True
bucket.patch()

# 3. Refresh and Verify
bucket.reload()
print(f"Versioning enabled status: {bucket.versioning_enabled}")

```

### Why this happens

* **Global Namespace:** Think of bucket names like website domains. Only one person can own `google.com`. Similarly, only one GCP project can own `archive-bucket`.
* **Idempotency:** When writing automation scripts, you want them to be "idempotent"—meaning they can run 10 times and produce the same result without crashing. Using `if not bucket.exists()` makes your script idempotent.

### Summary of common `Conflict` causes:

1. **Name is taken:** Someone else in the GCP universe is using that exact name.
2. **Double-run:** You already created the bucket in a previous attempt.
3. **Soft-deleted name:** You recently deleted a bucket with that name, and it hasn't been fully released by the system yet (usually takes a few seconds to a few minutes).

## Suspend Versioning on GCS Bucket

Greetings, Space Voyager! As you have successfully enabled versioning on a bucket in the previous task, it's now time to suspend versioning for a bucket. Being able to suspend versioning is crucial for managing costs, as versioning stores all versions of an object, including all writes and deletes. Let's put your skills to the test, shall we?

Important Note: Keep in mind that running scripts can modify the resources in our GCP simulator. To revert to the initial state, you can use the reset button located in the top right corner. However, be aware that resetting will erase any code changes. To preserve your code during a reset, consider copying it to the clipboard.

```python
import os
from google.cloud import storage
from google.auth.credentials import AnonymousCredentials
from google.api_core.exceptions import NotFound

# Initialize Cloud Storage client
client = storage.Client(
    project=os.environ.get('PROJECT_ID'),
    credentials=AnonymousCredentials(),
    client_options={'api_endpoint': os.environ.get('STORAGE_HOST')}
)

bucket_name = 'archive-bucket'
try:
    bucket = client.get_bucket(bucket_name)
except NotFound:
    bucket = client.create_bucket(bucket_name)

bucket.reload()
bucket.versioning_enabled = True
bucket.patch()

# Reload bucket after enabling versioning
bucket.reload()

# TO DO: Suspend the versioning
# TO DO: Print the bucket versioning status

```
import os
from google.cloud import storage
from google.auth.credentials import AnonymousCredentials
from google.api_core.exceptions import NotFound

# Initialize Cloud Storage client
client = storage.Client(
    project=os.environ.get('PROJECT_ID'),
    credentials=AnonymousCredentials(),
    client_options={'api_endpoint': os.environ.get('STORAGE_HOST')}
)

bucket_name = 'archive-bucket'
try:
    bucket = client.get_bucket(bucket_name)
except NotFound:
    bucket = client.create_bucket(bucket_name)

bucket.reload()
bucket.versioning_enabled = True
bucket.patch()

# Reload bucket after enabling versioning
bucket.reload()

# --- TODO FIXED: Suspend the versioning ---

# 1. Update the local property to False
bucket.versioning_enabled = False

# 2. Push the change to the GCS server
bucket.patch()

# 3. Reload to confirm the server accepted the change
bucket.reload()

# 4. Print the bucket versioning status
if bucket.versioning_enabled:
    print(f"Bucket '{bucket_name}' versioning status: ENABLED")
else:
    print(f"Bucket '{bucket_name}' versioning status: SUSPENDED")

## Google Cloud Storage Versioned Object Retrieval

We're on a journey with Google Cloud Storage versioning! You are provided with a partially completed Python script that interacts with a Cloud Storage bucket. The setup work has been done: a Cloud Storage bucket named cosmo-course-logos has been created, versioning is enabled for the bucket, and two different versions of a logo file have already been uploaded under the same key versioned-course-logo.jpg. Your objective is to execute the final steps of the script:

    Retrieve All Generation Numbers: Implement the section of the script that fetches all generation numbers for the object named versioned-course-logo.jpg. This step is crucial for identifying the available versions of the file.

    Download a Specific Version: Based on the generation numbers retrieved, modify the script to download the very first version of versioned-course-logo.jpg uploaded to the bucket. Note that, due to the reversed chronological order in which they are listed, the first version uploaded will be the last in the list. This downloaded file should be saved to the /usercode/FILESYSTEM/downloads folder, demonstrating your ability to access and restore specific versions of an object in a version-enabled Cloud Storage bucket.

Important Note: Running scripts can alter the file system's state or modify the resources in our GCP simulator. To revert to the initial state, you can use the reset button located in the top right corner. However, remember that resetting will erase any code changes. To preserve your code during a reset, consider copying it to the clipboard.

```python
import os
from google.cloud import storage
from google.auth.credentials import AnonymousCredentials
from google.api_core.exceptions import Conflict

# Initialize Cloud Storage client
client = storage.Client(
    project=os.environ.get('PROJECT_ID'),
    credentials=AnonymousCredentials(),
    client_options={'api_endpoint': os.environ.get('STORAGE_HOST')}
)

# Ensure that /usercode/FILESYSTEM/downloads folder exists
downloads_folder = "/usercode/FILESYSTEM/downloads"
os.makedirs(downloads_folder, exist_ok=True)

# Create Cloud Storage bucket called `cosmo-course-logos`
bucket_name = 'cosmo-course-logos'
try:
    bucket = client.create_bucket(bucket_name)
except Conflict:
    bucket = client.get_bucket(bucket_name)

bucket.reload()
bucket.versioning_enabled = True
bucket.patch()

# Reload bucket after enabling versioning
bucket.reload()

# Upload "/usercode/FILESYSTEM/assets/data-science-python-course-logo.jpg" to the bucket under object name "versioned-course-logo.jpg"
blob = bucket.blob('versioned-course-logo.jpg')
blob.upload_from_filename('/usercode/FILESYSTEM/assets/data-science-python-course-logo.jpg')

# Upload "/usercode/FILESYSTEM/assets/machine-learning-course-logo.jpg" to the bucket under object name "versioned-course-logo.jpg"
blob = bucket.blob('versioned-course-logo.jpg')
blob.upload_from_filename('/usercode/FILESYSTEM/assets/machine-learning-course-logo.jpg')

# TODO: Retrieve all generation numbers for the 'versioned-course-logo.jpg'. Remember, the first version uploaded will be the last in the list due to reverse order.

# TODO: Download the earliest version of 'versioned-course-logo.jpg' to the '/usercode/FILESYSTEM/downloads/' folder

```

To retrieve historical data in Google Cloud Storage, you must pivot from looking at the "Live" object to looking at the **Version History**. By passing `versions=True` to the `list_blobs` method, you can access the unique generation numbers that identify each iteration of your file.

Here is the completed script to retrieve and restore that first version:

```python
import os
from google.cloud import storage
from google.auth.credentials import AnonymousCredentials
from google.api_core.exceptions import Conflict

# Initialize Cloud Storage client
client = storage.Client(
    project=os.environ.get('PROJECT_ID'),
    credentials=AnonymousCredentials(),
    client_options={'api_endpoint': os.environ.get('STORAGE_HOST')}
)

# Ensure that /usercode/FILESYSTEM/downloads folder exists
downloads_folder = "/usercode/FILESYSTEM/downloads"
os.makedirs(downloads_folder, exist_ok=True)

# Create Cloud Storage bucket called `cosmo-course-logos`
bucket_name = 'cosmo-course-logos'
try:
    bucket = client.create_bucket(bucket_name)
except Conflict:
    bucket = client.get_bucket(bucket_name)

bucket.reload()
bucket.versioning_enabled = True
bucket.patch()

# Reload bucket after enabling versioning
bucket.reload()

# Upload version 1 (Data Science Logo)
blob = bucket.blob('versioned-course-logo.jpg')
blob.upload_from_filename('/usercode/FILESYSTEM/assets/data-science-python-course-logo.jpg')

# Upload version 2 (Machine Learning Logo)
blob = bucket.blob('versioned-course-logo.jpg')
blob.upload_from_filename('/usercode/FILESYSTEM/assets/machine-learning-course-logo.jpg')

# --- TODO FIXED: Retrieve all generation numbers ---

# We filter by prefix to get only versions for this specific file
blobs_iterator = client.list_blobs(bucket_name, versions=True, prefix='versioned-course-logo.jpg')
generations = [b.generation for b in blobs_iterator]

print(f"Detected {len(generations)} versions.")
for i, gen in enumerate(generations):
    print(f"Index {i}: Generation {gen}")

# --- TODO FIXED: Download the earliest version ---

# Since GCS lists them in reverse chronological order (newest first), 
# the earliest version is at the end of the list.
earliest_generation = generations[-1]
print(f"\nDownloading earliest version (Generation: {earliest_generation})...")

# Instantiate a blob pointing specifically to that generation
restored_blob = bucket.blob('versioned-course-logo.jpg', generation=earliest_generation)

# Define the local path and download
download_path = os.path.join(downloads_folder, 'restored-version-1.jpg')
restored_blob.download_to_filename(download_path)

print(f"Successfully restored to: {download_path}")

```

### Strategic Breakdown

* **`versions=True`**: This is the key that unlocks the archive. Without this, `list_blobs` only returns the single "Live" version.
* **Generation List Order**: GCS lists versions from newest to oldest. If you perform a "List" operation, the very first file you uploaded will be the `[-1]` element of your list.
* **Targeting the Generation**: When calling `bucket.blob()`, the `generation` parameter acts like a time-stamp. It ensures that the `download_to_filename` call pulls the specific historical bytes rather than the current logo.

This pattern is the foundation for creating "Restore" or "Rollback" features in any cloud-based application.

## Google Cloud Storage Object Versioning and Sequential Upload Management

Dive into the final task of managing space exploration data using Google Cloud Storage! Your mission is to craft a script that activates bucket versioning and sequentially uploads a series of cosmic image files. You are provided with three unique images intended for upload in a specific order. Each image will replace the previous one under the same object key, "course-logo.jpg", showcasing how versioning in Cloud Storage maintains each version of an object, even as it's updated or replaced. After uploading all images, retrieve and print all versions of the "course-logo.jpg" to demonstrate the history of changes made to this object.

Important Note: Running scripts can alter the filesystem's state or modify the resources in our GCP simulator. To revert to the initial state, you can use the reset button located in the top right corner. However, keep in mind that resetting will erase any code changes. To preserve your code during a reset, consider copying it to the clipboard.

```python
import os
from google.cloud import storage
from google.auth.credentials import AnonymousCredentials
from google.api_core.exceptions import Conflict

# Initialize Cloud Storage client
client = storage.Client(
    project=os.environ.get('PROJECT_ID'),
    credentials=AnonymousCredentials(),
    client_options={'api_endpoint': os.environ.get('STORAGE_HOST')}
)

# Create a new bucket specifically for our course logos.
bucket_name = 'educational-course-logos'
try:
    bucket = client.create_bucket(bucket_name)
except Conflict:
    bucket = client.get_bucket(bucket_name)

# TODO: Enable versioning on this newly created bucket.

# TODO: Upload 'prompt-engineering-course-logo.jpg' from '/usercode/FILESYSTEM/assets/' as 'course-logo.jpg'.

# TODO: Replace 'course-logo.jpg' by uploading 'machine-learning-course-logo.jpg' from the same directory.

# TODO: Finally, upload 'data-science-python-course-logo.jpg', replacing the current 'course-logo.jpg'.

# TODO: Print a confirmation after each upload, ensuring the process is clear.

# TODO: Retrieve and print all versions of 'course-logo.jpg', displaying the history of this object.
```

This final task demonstrates the "stacking" behavior of versioned objects. Even though you are using a single key—`course-logo.jpg`—Google Cloud Storage treats each upload as a distinct immutable resource identified by its **Generation Number**.

Here is the complete script to manage your sequential uploads and retrieve the version history:

```python
import os
from google.cloud import storage
from google.auth.credentials import AnonymousCredentials
from google.api_core.exceptions import Conflict

# Initialize Cloud Storage client
client = storage.Client(
    project=os.environ.get('PROJECT_ID'),
    credentials=AnonymousCredentials(),
    client_options={'api_endpoint': os.environ.get('STORAGE_HOST')}
)

# Create a new bucket specifically for our course logos.
bucket_name = 'educational-course-logos'
try:
    bucket = client.create_bucket(bucket_name)
except Conflict:
    bucket = client.get_bucket(bucket_name)

# 1. Enable versioning on this newly created bucket
bucket.versioning_enabled = True
bucket.patch()
bucket.reload()
print(f"Versioning enabled for {bucket_name}: {bucket.versioning_enabled}")

# List of images to upload sequentially
assets_path = '/usercode/FILESYSTEM/assets/'
image_files = [
    'prompt-engineering-course-logo.jpg',
    'machine-learning-course-logo.jpg',
    'data-science-python-course-logo.jpg'
]

# 2, 3, & 4. Sequentially upload images as the same object key
object_key = 'course-logo.jpg'

for image_file in image_files:
    blob = bucket.blob(object_key)
    blob.upload_from_filename(os.path.join(assets_path, image_file))
    # We reload to get the newly generated generation number
    blob.reload()
    print(f"Successfully uploaded {image_file} as {object_key}. Generation: {blob.generation}")

# 5. Retrieve and print all versions of 'course-logo.jpg'
print("\n--- Object Version History ---")
# Use versions=True to see non-current versions
blobs = client.list_blobs(bucket_name, versions=True, prefix=object_key)

for b in blobs:
    status = "LIVE" if b.metageneration == 1 else "ARCHIVED"
    print(f"Object: {b.name} | Generation: {b.generation} | Status: {status} | Updated: {b.updated}")

```

---

### Understanding the Sequential History

* **The Object Key vs. The Generation**: In the cloud console or a basic list, you only see one `course-logo.jpg`. However, behind the scenes, GCS keeps three distinct sets of binary data.
* **The `patch()` and `reload()` Cycle**: Always remember to `patch()` your configuration changes and `reload()` your blobs or buckets if you need to access system-generated metadata like the **Generation Number** or **Updated Timestamp**.
* **Archived Status**: In the history list, the version with the highest generation number is typically your "Live" version. Older versions are referred to as "Non-current" or "Archived."

Congratulations! You've successfully implemented a robust data versioning pipeline, ensuring that no cosmic data is lost during updates.